# Getting all Wastewater study datasets

Here we demonstrate how `mgnipy` can be used to build a cross-study taxonomic dataset for a given biome with rich sample metadata from MGnify and BioSamples in a few lines of code. 

First starting the session with a MGnipy client

In [2]:
from mgnipy import MGnipy

# configure session 
MG = MGnipy(cache_dir='wwtp')

# selecting the studies resource
studies_resource = MG.studies

# helper to see accepted search params for endpoint
studies_resource.describe_endpoint()

List all studies analysed by MGnify

MGnify studies inherit directly from studies (or projects) in ENA.

Supported parameters:
- order: ListMgnifyStudiesOrderType0 | None | Unset
- biome_lineage: None | str | Unset The lineage to match, including all descendant biomes
- has_analyses_from_pipeline: None | PipelineVersions | Unset If set, will only show studies with analyses from the specified MGnify pipeline version
- search: None | str | Unset Search within study titles and accessions
- page: int | Unset Default: 1.
- page_size: int | None | Unset


now using mgnifier to build and then execute the query set

In [3]:
# preparing query set
wwtp_studies = studies_resource(biome_lineage='root:Engineered:Wastewater')
# helper to preview the query set prior to fetch
wwtp_studies.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=1
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=2
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=3
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=4
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=5
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=6
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=7
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=8


In [4]:
# now actually executing queries
async with MG: 
    # get all 8 pages of records from the list endpoint
    await wwtp_studies.aget_all()
    # enrich the records with study metadata from detail endpoint
    await wwtp_studies.aenrich_details()

Enriching study details: 100%|██████████| 189/189 [00:00<00:00, 359.19it/s]


In [5]:
# taking a look at metdata so far 
wwtp_studies.metadata.to_pandas(expand_nested_dicts=True).head()

,accession,ena_accessions,title,updated_at,downloads,first_accession,biome__biome_name,biome__lineage
0,MGYS00000555,"[ERP012888, PRJEB11494]",Wastewater metagenomics.,2026-05-28T15:46:49.653000+00:00,"[{'file_type': 'tsv', 'download_type': 'Functi...",ERP012888,Wastewater,root:Engineered:Wastewater
1,MGYS00003379,"[ERP111146, PRJEB28884]",EMG produced TPA metagenomics assembly of the ...,2026-05-28T15:46:54.002000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP111146,Activated Sludge,root:Engineered:Wastewater:Activated Sludge
2,MGYS00005279,"[ERP112197, PRJEB29847]",EMG produced TPA metagenomics assembly of the ...,2026-05-28T15:47:00.642000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP112197,Wastewater,root:Engineered:Wastewater
3,MGYS00004985,"[ERP112879, PRJEB30426]",EMG produced TPA metagenomics assembly of the ...,2026-05-28T15:46:57.144000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP112879,Activated Sludge,root:Engineered:Wastewater:Activated Sludge
4,MGYS00001764,"[SRP009669, PRJNA78967]",Bacterial community composition in coking wast...,2026-05-28T15:46:51.860000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",SRP009669,Industrial wastewater,root:Engineered:Wastewater:Industrial wastewater


exploring the MGazine of datasets for the studies

In [6]:
# getting the magazine of datasets
MZ = wwtp_studies.datasets
# filtering to taxonomic of interest
filtered_MZ = MZ['v4_1']['Taxonomic assignments SSU']
# with helpers 
taxo_mz = filtered_MZ.taxonomic 

TaxaMGazine containing:
- MGnify pipeline versions: ['v4', 'v4_1', 'v5']
- Number of downloads: 123
- Short descriptions: ['Taxonomic assignments SSU']
- Nonempty metadata sets: .mgnify_studies
-----------------------
Next steps: Use `.load()` to initialize.



In [7]:
# lazy loading the datasets
taxo_mz.load()

optionally we use MGnetizers to collect additional information from MGnify given accessions

In [8]:
# collecting run/assembly metadata for given accessions
run_accs = [x for x in taxo_mz.runs_accessions if not x.startswith('ERZ')]
assembly_accs = [x for x in taxo_mz.runs_accessions if x.startswith('ERZ')]
# init mgnetizer to collect 
mnet_run = MG.mgnetizer(resource='run', all_ids=run_accs)
mnet_assembly = MG.mgnetizer(resource='assembly', all_ids=assembly_accs)
# now executing the requests to the detail endpoints
async with MG: 
    await mnet_run.aenrich(limit=None)
    await mnet_assembly.aenrich(limit=None)

Enriching metadata from MGnify: 100%|██████████| 528/528 [00:00<00:00, 53.33it/s]


In [9]:
# passing the additional metadata to the Mgazine of taxonomic datasets
taxo_mz.mgnify_runs = mnet_run.metadata.to_list() + mnet_assembly.metadata.to_list()

,experiment_type,instrument_model,instrument_platform,accession,sample_accession,study_accession,updated_at,run_accession,status,sample__accession,sample__ena_accessions,sample__sample_title,sample__biome,sample__updated_at,study__accession,study__ena_accessions,study__title,study__updated_at,study__biome.biome_name,study__biome.lineage
0,Amplicon,454 GS FLX,LS454,DRR046685,SAMD00041314,MGYS00005741,NaN,NaN,NaN,SAMD00041314,"[SAMD00041314, DRS050256]",WL10_018,NaN,2026-04-30T16:33:28.429000+00:00,MGYS00005741,"[DRP003823, PRJDB4240]",WL Reactor published in ME,2026-05-28T15:47:00.074000+00:00,Activated Sludge,root:Engineered:Wastewater:Activated Sludge
1,Amplicon,454 GS FLX,LS454,DRR046680,SAMD00041309,MGYS00005741,NaN,NaN,NaN,SAMD00041309,"[DRS050251, SAMD00041309]",WL10_007,NaN,2026-04-30T16:33:29.770000+00:00,MGYS00005741,"[DRP003823, PRJDB4240]",WL Reactor published in ME,2026-05-28T15:47:00.074000+00:00,Activated Sludge,root:Engineered:Wastewater:Activated Sludge
2,Amplicon,454 GS FLX,LS454,DRR046681,SAMD00041310,MGYS00005741,NaN,NaN,NaN,SAMD00041310,"[SAMD00041310, DRS050252]",WL10_009,NaN,2026-04-30T16:33:29.101000+00:00,MGYS00005741,"[DRP003823, PRJDB4240]",WL Reactor published in ME,2026-05-28T15:47:00.074000+00:00,Activated Sludge,root:Engineered:Wastewater:Activated Sludge
3,Amplicon,454 GS FLX,LS454,DRR046686,SAMD00041315,MGYS00005741,NaN,NaN,NaN,SAMD00041315,"[SAMD00041315, DRS050257]",WL10_021,NaN,2026-04-30T16:33:27.744000+00:00,MGYS00005741,"[DRP003823, PRJDB4240]",WL Reactor published in ME,2026-05-28T15:47:00.074000+00:00,Activated Sludge,root:Engineered:Wastewater:Activated Sludge
4,Amplicon,454 GS FLX,LS454,DRR046683,SAMD00041312,MGYS00005741,NaN,NaN,NaN,SAMD00041312,"[DRS050254, SAMD00041312]",WL10_014,NaN,2026-04-30T16:32:56.389000+00:00,MGYS00005741,"[DRP003823, PRJDB4240]",WL Reactor published in ME,2026-05-28T15:47:00.074000+00:00,Activated Sludge,root:Engineered:Wastewater:Activated Sludge


optionally can also use BioSampler helper to collect additional sample metadata from BioSamples

In [10]:
# # the sample accessions to use
# sample_ids = taxo_mz.mgnify_runs.to_pandas()['sample_accession'].unique()
# # init biosampler to collect
# bios = MG.biosampler(sample_ids)
# # actually executing the requests
# async with MG: 
#     await bios.aenrich(limit=None)
# # passing the matadata back to MGazine
# taxo_mz.biosamples_metadata = bios.metadata.to_list(drop_duplicates=True)
# # taking a look 
# print(taxo_mz)

from the TaxaMGazine we can get an annotated dataframe with the observation metadata and taxonomic metadata (i.e., taxonomic ranks)

In [11]:
# convert to annotated dataframe
an_df = taxo_mz.to_anndata()
# demo adding a layer with filled in zeros 
an_df.layers['filled_zeros'] = an_df.to_df().fillna(0)
# exporting to h5ad file 
an_df.obs = an_df.obs.astype(str) #workaround for h5ad export issue with mixed types in obs
an_df.write_h5ad('wwtp_biome.h5ad')

as an example here we demo how to use the dataset in a new script:

In [12]:
import anndata as ad 
# read in data
back = ad.read_h5ad('wwtp_biome.h5ad')
# check it out
back

AnnData object with n_obs × n_vars = 1875 × 12120
    obs: 'experiment_type', 'instrument_model', 'instrument_platform', 'sample_accession', 'study_accession', 'updated_at', 'run_accession', 'status', 'sample__accession', 'sample__ena_accessions', 'sample__sample_title', 'sample__biome', 'sample__updated_at', 'study__accession', 'study__ena_accessions', 'study__title', 'study__updated_at', 'study__biome.biome_name', 'study__biome.lineage'
    var: 'Superkingdom', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species'
    layers: 'filled_zeros'
